# Lab 02 - PySpark Transformation Lab
## Using Databricks Sample Data (NYC Taxi + TPC-H)

**Durée:** ~60 minutes

Ce lab couvre les transformations PySpark fondamentales :
- Lecture de données depuis le catalogue `samples`
- Filtrage avec conditions multiples
- Ajout de colonnes calculées (CASE WHEN, extraction de dates)
- Jointures entre tables
- Agrégations (groupBy, fonctions d'agrégation)
- Fonctions de fenêtre (rank, running total, lag/lead)

**Aucune configuration requise** — les données sont pré-chargées dans `samples.nyctaxi` et `samples.tpch`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, when, sum as spark_sum, count, avg, max as spark_max, min as spark_min,
    round as spark_round, year, month, hour, date_format, desc, asc,
    row_number, rank, dense_rank, lag, lead
)
from pyspark.sql.window import Window

print("Libraries imported.")

---
## Section 1: Lecture des données

> Les données proviennent du catalogue `samples` intégré à Databricks.

### Exercise 1.1 - Lire la table NYC Taxi Trips

In [ ]:
# Exercise 1.1 - Read the NYC Taxi trips table
# HINT: spark.read.table("catalog.schema.table")

trips_df = spark.read.table("samples.nyctaxi.trips")

print(f"Trips count: {trips_df.count()}")
trips_df.printSchema()
display(trips_df.limit(5))

In [ ]:
# Verification 1.1
assert trips_df.count() > 0, "trips_df is empty"
assert "fare_amount" in trips_df.columns, "Missing fare_amount column"
print("Exercise 1.1 passed!")

### Exercise 1.2 - Lire les tables TPC-H (Orders + Customers)

In [ ]:
# Exercise 1.2 - Read TPC-H orders and customers tables

orders_df = spark.read.table("samples.tpch.orders")
customers_df = spark.read.table("samples.tpch.customer")

print(f"Orders: {orders_df.count()} rows")
print(f"Customers: {customers_df.count()} rows")
orders_df.printSchema()

In [ ]:
# Verification 1.2
assert orders_df.count() > 0, "orders_df is empty"
assert customers_df.count() > 0, "customers_df is empty"
print("Exercise 1.2 passed!")

---
## Section 2: Filtrage

> Sélectionner des sous-ensembles de données avec des conditions.

### Exercise 2.1 - Filtrer les trajets longs et chers

In [ ]:
%sql
-- Équivalent SQL pour comparaison
SELECT * FROM samples.nyctaxi.trips
WHERE fare_amount > 20 AND trip_distance > 5
LIMIT 10

In [ ]:
# Exercise 2.1 - Filter: fare > $20 AND distance > 5 miles
# TODO: Fill in the filter conditions

long_expensive_trips = trips_df.filter(
    (col("fare_amount") > 20) &
    (col("trip_distance") > 5)
)

print(f"Long expensive trips: {long_expensive_trips.count()}")
display(long_expensive_trips.limit(10))

In [ ]:
# Verification 2.1
assert long_expensive_trips.count() > 0, "No trips found — check filter conditions"
print("Exercise 2.1 passed!")

### Exercise 2.2 - Filtrer avec IN et sélection de colonnes

In [ ]:
%sql
-- Équivalent SQL
SELECT o_orderkey, o_custkey, o_totalprice, o_orderstatus
FROM samples.tpch.orders
WHERE o_orderstatus IN ('O', 'P')
ORDER BY o_totalprice DESC
LIMIT 10

In [ ]:
# Exercise 2.2 - Filter orders with status IN ('O', 'P'), select columns, sort
# TODO: Fill in isin values and column names

filtered_orders = orders_df \
    .filter(col("o_orderstatus").isin("O", "P")) \
    .select("o_orderkey", "o_custkey", "o_totalprice", "o_orderstatus") \
    .orderBy(col("o_totalprice").desc())

print(f"Filtered orders: {filtered_orders.count()}")
display(filtered_orders.limit(10))

In [ ]:
# Verification 2.2
assert filtered_orders.count() > 0
assert set(filtered_orders.select("o_orderstatus").distinct().toPandas()["o_orderstatus"].tolist()).issubset({"O", "P"})
print("Exercise 2.2 passed!")

---
## Section 3: Colonnes calculées

> Ajouter des colonnes dérivées avec des expressions conditionnelles et des extractions de dates.

### Exercise 3.1 - Colonne conditionnelle (CASE WHEN) : fare_per_mile

In [ ]:
%sql
-- Équivalent SQL
SELECT *, 
  CASE WHEN trip_distance > 0 THEN ROUND(fare_amount / trip_distance, 2)
       ELSE 0 
  END AS fare_per_mile
FROM samples.nyctaxi.trips
LIMIT 10

In [ ]:
# Exercise 3.1 - Add fare_per_mile column using when/otherwise
# HINT: when(condition, value).otherwise(other_value)
# Handle division by zero when trip_distance = 0

trips_with_fare = trips_df.withColumn(
    "fare_per_mile",
    when(col("trip_distance") > 0, spark_round(col("fare_amount") / col("trip_distance"), 2)).otherwise(0)
)

display(trips_with_fare.select("trip_distance", "fare_amount", "fare_per_mile").limit(10))

In [ ]:
# Verification 3.1
assert "fare_per_mile" in trips_with_fare.columns
zero_distance = trips_with_fare.filter(col("trip_distance") == 0).select("fare_per_mile").first()
if zero_distance:
    assert zero_distance[0] == 0, "fare_per_mile should be 0 when distance is 0"
print("Exercise 3.1 passed!")

### Exercise 3.2 - Extraction de date : heure et durée

In [ ]:
%sql
-- Équivalent SQL
SELECT 
  tpep_pickup_datetime,
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  ROUND((UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60, 1) AS trip_duration_min
FROM samples.nyctaxi.trips
LIMIT 10

In [ ]:
# Exercise 3.2 - Extract pickup hour + calculate trip duration in minutes
# HINT: hour() for hour extraction
# HINT: (unix_timestamp(dropoff) - unix_timestamp(pickup)) / 60 for duration

trips_time = trips_df \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("trip_duration_min", 
        spark_round(
            (F.unix_timestamp(col("tpep_dropoff_datetime")) - F.unix_timestamp(col("tpep_pickup_datetime"))) / 60, 1
        )
    )

display(trips_time.select("tpep_pickup_datetime", "pickup_hour", "trip_duration_min").limit(10))

In [ ]:
# Verification 3.2
assert "pickup_hour" in trips_time.columns
assert "trip_duration_min" in trips_time.columns
sample_hour = trips_time.select("pickup_hour").first()[0]
assert 0 <= sample_hour <= 23, "pickup_hour should be between 0 and 23"
print("Exercise 3.2 passed!")

---
## Section 4: Jointures

> Combiner des tables avec des jointures pour enrichir les données.

### Exercise 4.1 - Jointure Orders ↔ Customers

In [ ]:
%sql
-- Équivalent SQL
SELECT o.o_orderkey, o.o_totalprice, c.c_name, c.c_mktsegment
FROM samples.tpch.orders o
INNER JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
LIMIT 10

In [ ]:
# Exercise 4.1 - Inner join orders with customers
# TODO: Fill in the join condition

orders_customers = orders_df.join(
    customers_df,
    orders_df["o_custkey"] == customers_df["c_custkey"],
    "inner"
)

print(f"Joined rows: {orders_customers.count()}")
display(orders_customers.select("o_orderkey", "o_totalprice", "c_name", "c_mktsegment").limit(10))

In [ ]:
# Verification 4.1
assert orders_customers.count() > 0
assert "c_name" in orders_customers.columns
print("Exercise 4.1 passed!")

### Exercise 4.2 - Jointure avec sélection de colonnes

In [ ]:
# Exercise 4.2 - Join lineitem with orders, select specific columns
# Lire lineitem, joindre avec orders, garder : l_orderkey, l_extendedprice, l_quantity, o_orderdate, o_orderstatus

lineitem_df = spark.read.table("samples.tpch.lineitem")

line_orders = lineitem_df.join(
    orders_df,
    lineitem_df["l_orderkey"] == orders_df["o_orderkey"],
    "inner"
).select("l_orderkey", "l_extendedprice", "l_quantity", "o_orderdate", "o_orderstatus")

print(f"Line-order rows: {line_orders.count()}")
display(line_orders.limit(10))

In [ ]:
# Verification 4.2
assert line_orders.count() > 0
assert "o_orderdate" in line_orders.columns
print("Exercise 4.2 passed!")

---
## Section 5: Agrégations

> Résumer les données avec groupBy et fonctions d'agrégation.

### Exercise 5.1 - Fare moyenne et nombre de trajets par zone

In [ ]:
%sql
-- Équivalent SQL
SELECT pickup_zip, 
       COUNT(*) AS trip_count, 
       ROUND(AVG(fare_amount), 2) AS avg_fare,
       ROUND(SUM(fare_amount), 2) AS total_fare
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
ORDER BY trip_count DESC
LIMIT 10

In [ ]:
# Exercise 5.1 - Aggregate: count, avg fare, total fare by pickup_zip
# TODO: Fill in the aggregation functions

zone_stats = trips_df.groupBy("pickup_zip").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(spark_sum("fare_amount"), 2).alias("total_fare")
).orderBy(col("trip_count").desc())

display(zone_stats.limit(10))

In [ ]:
# Verification 5.1
assert "trip_count" in zone_stats.columns
assert "avg_fare" in zone_stats.columns
print("Exercise 5.1 passed!")

### Exercise 5.2 - Revenu par segment de marché (TPC-H)

In [ ]:
%sql
-- Équivalent SQL
SELECT c.c_mktsegment, 
       COUNT(*) AS order_count,
       ROUND(SUM(o.o_totalprice), 2) AS total_revenue,
       ROUND(AVG(o.o_totalprice), 2) AS avg_order_value
FROM samples.tpch.orders o
JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
GROUP BY c.c_mktsegment
ORDER BY total_revenue DESC

In [ ]:
# Exercise 5.2 - Revenue by market segment
# Use orders_customers from Exercise 4.1

segment_revenue = orders_customers.groupBy("c_mktsegment").agg(
    count("*").alias("order_count"),
    spark_round(spark_sum("o_totalprice"), 2).alias("total_revenue"),
    spark_round(avg("o_totalprice"), 2).alias("avg_order_value")
).orderBy(col("total_revenue").desc())

display(segment_revenue)

In [ ]:
# Verification 5.2
assert segment_revenue.count() == 5, "Should have 5 market segments"
print("Exercise 5.2 passed!")

---
## Section 6: Fonctions de fenêtre

> Classement, totaux cumulés, et comparaison avec les lignes précédentes/suivantes.

### Exercise 6.1 - Classement des clients par solde (dense_rank)

In [ ]:
%sql
-- Équivalent SQL
SELECT c_name, c_mktsegment, c_acctbal,
       DENSE_RANK() OVER (PARTITION BY c_mktsegment ORDER BY c_acctbal DESC) AS balance_rank
FROM samples.tpch.customer
QUALIFY balance_rank <= 3

In [ ]:
# Exercise 6.1 - Rank customers by account balance within market segment
# TODO: Fill in the window spec and ranking function
# HINT: Window.partitionBy(...).orderBy(...)

window_spec = Window.partitionBy("c_mktsegment").orderBy(col("c_acctbal").desc())

ranked_customers = customers_df \
    .withColumn("balance_rank", dense_rank().over(window_spec)) \
    .filter(col("balance_rank") <= 3) \
    .select("c_name", "c_mktsegment", "c_acctbal", "balance_rank")

display(ranked_customers.orderBy("c_mktsegment", "balance_rank"))

In [ ]:
# Verification 6.1
assert "balance_rank" in ranked_customers.columns
assert ranked_customers.filter(col("balance_rank") > 3).count() == 0
print("Exercise 6.1 passed!")

### Exercise 6.2 - Total cumulé des commandes par client

In [ ]:
# Exercise 6.2 - Running total of order amounts by customer
# HINT: Window.partitionBy("o_custkey").orderBy("o_orderdate").rowsBetween(Window.unboundedPreceding, Window.currentRow)

window_running = Window.partitionBy("o_custkey").orderBy("o_orderdate") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

orders_running = orders_df \
    .withColumn("running_total", spark_round(spark_sum("o_totalprice").over(window_running), 2)) \
    .select("o_custkey", "o_orderdate", "o_totalprice", "running_total")

# Show for one customer
display(orders_running.filter(col("o_custkey") == 1).orderBy("o_orderdate"))

In [ ]:
# Verification 6.2
assert "running_total" in orders_running.columns
print("Exercise 6.2 passed!")

### Exercise 6.3 - Comparer avec la commande précédente (Lag / Lead)

In [ ]:
# Exercise 6.3 - Compare each order with previous and next order amount
# HINT: lag("column", 1) for previous, lead("column", 1) for next

window_seq = Window.partitionBy("o_custkey").orderBy("o_orderdate")

orders_compare = orders_df \
    .withColumn("prev_order_amount", lag("o_totalprice", 1).over(window_seq)) \
    .withColumn("next_order_amount", lead("o_totalprice", 1).over(window_seq)) \
    .withColumn("change_from_prev", 
        spark_round(col("o_totalprice") - col("prev_order_amount"), 2)
    ) \
    .select("o_custkey", "o_orderdate", "o_totalprice", "prev_order_amount", "next_order_amount", "change_from_prev")

display(orders_compare.filter(col("o_custkey") == 1).orderBy("o_orderdate"))

In [ ]:
# Verification 6.3
assert "prev_order_amount" in orders_compare.columns
assert "next_order_amount" in orders_compare.columns
print("Exercise 6.3 passed!")

---
## Section 7: Challenge

> Combiner toutes les techniques dans un pipeline multi-étapes.

**Objectif:** Créer un rapport qui montre, pour chaque segment de marché, les 3 meilleurs clients par montant total commandé, avec leur rang et le pourcentage du segment.

In [ ]:
# Challenge: Multi-step pipeline
# 1. Join orders with customers
# 2. Aggregate total spent per customer per segment
# 3. Rank customers within each segment
# 4. Filter top 3 per segment
# 5. Add percentage of segment total

# Step 1 + 2: Join and aggregate
customer_totals = orders_df.join(
    customers_df,
    orders_df["o_custkey"] == customers_df["c_custkey"],
    "inner"
).groupBy("c_custkey", "c_name", "c_mktsegment").agg(
    spark_round(spark_sum("o_totalprice"), 2).alias("total_spent"),
    count("o_orderkey").alias("order_count")
)

# Step 3: Rank within segment
w = Window.partitionBy("c_mktsegment").orderBy(col("total_spent").desc())
ranked = customer_totals.withColumn("segment_rank", dense_rank().over(w))

# Step 4: Top 3
top3 = ranked.filter(col("segment_rank") <= 3)

# Step 5: Percentage of segment
segment_totals = customer_totals.groupBy("c_mktsegment").agg(
    spark_sum("total_spent").alias("segment_total")
)

result = top3.join(segment_totals, "c_mktsegment") \
    .withColumn("pct_of_segment", spark_round(col("total_spent") / col("segment_total") * 100, 2)) \
    .select("c_mktsegment", "segment_rank", "c_name", "total_spent", "order_count", "pct_of_segment") \
    .orderBy("c_mktsegment", "segment_rank")

display(result)

In [ ]:
# Verification Challenge
assert result.count() == 15, f"Expected 15 rows (3 per 5 segments), got {result.count()}"
assert "pct_of_segment" in result.columns
print("Challenge passed! Well done!")

---
## Lab Complete!

Vous avez pratiqué :
- ✅ Lecture de tables depuis le catalogue Unity Catalog
- ✅ Filtrage avec conditions multiples et IN
- ✅ Colonnes calculées (CASE WHEN, extraction de dates)
- ✅ Jointures (inner join, multi-table)
- ✅ Agrégations (groupBy, count, sum, avg)
- ✅ Fonctions de fenêtre (dense_rank, running total, lag/lead)
- ✅ Pipeline multi-étapes combinant toutes les techniques